# 00 - Comprendre le projet LAAFI_AI

Objectif : comprendre, sans jargon inutile, comment on passe d'une image histopathologique a une prediction binaire : `metastase` ou `normal`.

> Ce notebook est pedagogique. Il ne lance pas l'entrainement lourd.

## 1. Le probleme medical

PatchCamelyon contient des petits patchs de tissus colores H&E. Chaque image fait 96 x 96 pixels. Le label vaut :

- `0` : pas de metastase au centre du patch ;
- `1` : presence de tissu tumoral/metastatique.

Le modele apprend une fonction :

```text
image histopathologique -> ResNet50 -> probabilite de metastase
```

## 2. Schema global du pipeline

```mermaid
flowchart LR
    A[PatchCamelyon] --> B[DataLoader PyTorch]
    B --> C[Augmentations]
    C --> D[ResNet50 pre-entraine]
    D --> E[Probabilite metastase]
    E --> F[Metriques: AUC, sensibilite, specificite]
    E --> G[Grad-CAM]
```

In [ ]:
import matplotlib.pyplot as plt

steps = [
    "Dataset\nPCam",
    "Transformations\nresize + normalisation",
    "ResNet50\nImageNet",
    "Score\n0 a 1",
    "Evaluation\nAUC + ROC + PR",
    "Grad-CAM\ninterpretabilite",
]

fig, ax = plt.subplots(figsize=(14, 3))
ax.axis("off")
for i, step in enumerate(steps):
    ax.text(i, 0.5, step, ha="center", va="center", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.35", fc="#f6f7fb", ec="#4051b5"))
    if i < len(steps) - 1:
        ax.annotate("", xy=(i + 0.42, 0.5), xytext=(i + 0.78, 0.5),
                    arrowprops=dict(arrowstyle="->", lw=1.5))
ax.set_xlim(-0.7, len(steps) - 0.3)
ax.set_ylim(0, 1)
plt.show()

## 3. Pourquoi ResNet50 ?

ResNet50 est un reseau de neurones convolutif. Il a deja appris des motifs visuels generaux sur ImageNet : bords, textures, formes. Pour PCam, on remplace sa derniere couche pour produire une seule valeur : la probabilite de metastase.

Strategie simple :

1. Geler le backbone et entrainer seulement la tete de classification.
2. Degeler le dernier bloc `layer4` pour affiner les representations sur les tissus histologiques.

## 4. Les metriques importantes

- **AUC ROC** : capacite globale a separer positifs et negatifs.
- **Sensibilite** : parmi les vrais positifs, combien sont detectes ? Important pour ne pas rater une metastase.
- **Specificite** : parmi les vrais negatifs, combien sont correctement rejetes ? Important pour limiter les fausses alertes.
- **Precision** : parmi les predictions positives, combien sont vraiment positives ?

Pour un portfolio medical, il faut expliquer ces compromis, pas seulement afficher une accuracy.

In [ ]:
import numpy as np
from sklearn.metrics import roc_curve, precision_recall_curve, auc, average_precision_score

labels = np.array([0, 0, 0, 1, 1, 1])
scores = np.array([0.05, 0.25, 0.4, 0.55, 0.8, 0.95])

fpr, tpr, _ = roc_curve(labels, scores)
precision, recall, _ = precision_recall_curve(labels, scores)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(fpr, tpr, marker="o")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set_title(f"ROC AUC = {auc(fpr, tpr):.2f}")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")

axes[1].plot(recall, precision, marker="o")
axes[1].set_title(f"Average Precision = {average_precision_score(labels, scores):.2f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
plt.tight_layout()
plt.show()

## 5. Pieges classiques

- Ne pas melanger les splits officiels : sinon fuite de donnees.
- Ne pas charger tout le dataset en RAM.
- Ne pas croire aveuglement une AUC trop parfaite.
- Documenter les limites : ce n'est pas un outil clinique.